# E040 — SR-MPGD checkpoint frontier

E040 garde **γ=1000**, la loss `scanaware_v2` d'E039 et évalue chaque checkpoint `i0..i8` pour cinq rayons `0.150..0.250`. Le gagnant est le meilleur **checkpoint sûr**, pas forcément le dernier.

Le CNN E016, lorsqu'il est promu `research_usable` par sa propre model card, est affiché comme score secondaire. QR-Verify reste la vérité terrain logicielle.


In [ ]:
from __future__ import annotations
import json, os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image as DisplayImage, Markdown, display
RESULTS_DIR = Path(os.environ.get('E040_RESULTS_DIR','/data/e040-srmpgd-checkpoint-frontier-v1'))
print('E040 résultats :', RESULTS_DIR)


## 1. Verdict

In [ ]:
verdict = json.loads((RESULTS_DIR/'verdict.json').read_text(encoding='utf-8'))
display(Markdown('**E040 terminé : résultats disponibles.**'))
display(verdict)


## 2. Tous les checkpoints — vrai SSR + esthétique

In [ ]:
df = pd.read_csv(RESULTS_DIR/'checkpoint-comparison.csv')
cols = ['checkpoint','radius','iteration','gamma','qr_verify_exact_presets','ssr','original_exact','full_module_error_count','lpips','latent_delta_rms','clip_score','clip_aesthetic','hpsv2_1','surrogate_mean_success_probability','visual_guard_pass','acceptance_reason']
cols=[c for c in cols if c in df.columns]
display(df[cols].sort_values(['qr_verify_exact_presets','visual_guard_pass','lpips'], ascending=[False,False,True]).reset_index(drop=True))


## 3. Pourquoi un checkpoint est-il rejeté par la garde visuelle ?

In [ ]:
raw = json.loads((RESULTS_DIR/'checkpoint-comparison.json').read_text(encoding='utf-8'))
guard_rows=[]
for row in raw:
    checks=row.get('visual_guard_checks') or {}
    guard_rows.append({'checkpoint':row['checkpoint'],'SSR /37':row['qr_verify_exact_presets'],'LPIPS':row['lpips'],'safe':row['visual_guard_pass'],**checks})
guards=pd.DataFrame(guard_rows)
display(guards.sort_values(['SSR /37','safe'],ascending=[False,False]).reset_index(drop=True))


## 4. Meilleur checkpoint par rayon

In [ ]:
best = pd.DataFrame(json.loads((RESULTS_DIR/'best-checkpoint-per-radius.json').read_text(encoding='utf-8')))
if not best.empty:
    display(best[['checkpoint','radius','iteration','qr_verify_exact_presets','full_module_error_count','lpips','visual_guard_pass']])


## 5. SSR selon rayon et itération

In [ ]:
fig, ax = plt.subplots(figsize=(11,6))
for radius, group in df.groupby('radius'):
    group=group.sort_values('iteration')
    ax.plot(group['iteration'],group['qr_verify_exact_presets'],marker='o',label=f'r={radius:.3f}')
ax.set_xlabel('checkpoint SR-MPGD')
ax.set_ylabel('QR-Verify exact presets / 37')
ax.set_title('E040 — le meilleur checkpoint n’est pas forcément le dernier')
ax.legend()
plt.tight_layout(); plt.show()


## 6. SSR ↔ LPIPS

In [ ]:
fig, ax = plt.subplots(figsize=(9,6))
for radius, group in df.groupby('radius'):
    ax.scatter(group['lpips'],group['qr_verify_exact_presets'],label=f'r={radius:.3f}')
ax.set_xlabel('LPIPS vs parent'); ax.set_ylabel('SSR /37'); ax.set_title('E040 — frontier scan / esthétique')
ax.legend(); plt.tight_layout(); plt.show()


## 7. Pipeline complète en une image

In [ ]:
display(DisplayImage(filename=str(RESULTS_DIR/'pipeline/full-pipeline-contact-sheet.png'), width=1500))


## 8. Trajectoire complète du gagnant

In [ ]:
winner_recipe=verdict['research_winner_recipe']
winner_iteration=int(verdict['winner_iteration'])
for i in range(9):
    row=df[(df.method==winner_recipe)&(df.iteration==i)].iloc[0]
    display(Markdown(f"### i{i} — SSR={int(row.qr_verify_exact_presets)}/37 · LPIPS={row.lpips:.4f} · safe={row.visual_guard_pass}"))
    display(DisplayImage(filename=str(RESULTS_DIR/winner_recipe/'images'/f'iteration-{i:03d}.png'), width=720))
    if i==winner_iteration: display(Markdown('**← CHECKPOINT FINAL SÉLECTIONNÉ**'))


## 9. Modèle E016 — score des checkpoints

In [ ]:
status=json.loads((RESULTS_DIR/'e016-surrogate-status.json').read_text(encoding='utf-8'))
display(status)
if status.get('research_usable'):
    scores=pd.DataFrame([{'checkpoint':k,**v} for k,v in json.loads((RESULTS_DIR/'e016-surrogate-scores.json').read_text(encoding='utf-8')).items()])
    display(scores.sort_values('mean_success_probability',ascending=False).head(20))
else:
    display(Markdown('E016 n’est pas utilisé pour départager tant que sa propre card ne le marque pas `research_usable`.'))


## 10. Advisor E026/E031 — recommandation prospective

In [ ]:
advisor=json.loads((RESULTS_DIR/'advisor-preview.json').read_text(encoding='utf-8'))
display(advisor)


## 11. E039 vs E040

In [ ]:
control=json.loads((RESULTS_DIR/'e039-control.json').read_text(encoding='utf-8'))
winner=df[df.checkpoint==verdict['research_winner_checkpoint']].iloc[0]
summary=pd.DataFrame([
 {'method':'E039 safe winner','SSR /37':control['verdict'].get('winner_ssr_exact_presets'),'LPIPS':control['winner'].get('lpips'),'MER':control['winner'].get('full_module_error_count')},
 {'method':'E040 best checkpoint','SSR /37':winner.qr_verify_exact_presets,'LPIPS':winner.lpips,'MER':winner.full_module_error_count},
])
display(summary)
